# Notebook 06 — Feedback & Flagging

**Prerequisites:** Notebook 05 completed (conversations exist in DB)

**What to validate:**
- Thumbs-up as functional user (weight=1.0)
- Thumbs-up as Global Auditor (weight=2.0)
- Thumbs-down requires comment
- Thumbs-down flags conversation + writes snapshot
- Weighted aggregation query

In [1]:
import sys
sys.path.insert(0, '..')

import uuid
from sqlalchemy import select

from backend.db.session import get_session
from backend.db.models import User, Message, Conversation, ResponseFormat
from backend.feedback import feedback_service, flag_service
from backend.core.exceptions import FeedbackError

print('Imports OK')

Imports OK


## 1. Setup — Find an Assistant Message to Rate

In [2]:
async def get_test_message():
    async with get_session() as session:
        # Find an assistant message from any conversation
        from backend.db.models import MessageRole
        result = await session.execute(
            select(Message).where(Message.role == MessageRole.assistant).limit(1)
        )
        msg = result.scalar_one_or_none()
        if msg:
            print(f'Found message: {msg.msg_id}')
            print(f'Content preview: {msg.content[:100]}...')
            return msg.msg_id, msg.conv_id
        else:
            print('No messages found. Run notebook 05 first to create conversations.')
            return None, None

test_msg_id, test_conv_id = await get_test_message()

No messages found. Run notebook 05 first to create conversations.


## 2. Thumbs-Up as Functional User (weight = 1.0)

In [3]:
if test_msg_id:
    async def test_thumbs_up_functional():
        async with get_session() as session:
            it_user = (await session.execute(select(User).where(User.email == 'it.user@example.com'))).scalar_one()
            
            feedback = await feedback_service.record_feedback(
                session=session,
                msg_id=test_msg_id,
                rating=5,
                given_by=it_user.user_id,
                comments=None,
            )
            print(f'Feedback recorded: rating={feedback.rating}, weight={feedback.weight}')
            assert feedback.weight == 1.0, f'Expected 1.0, got {feedback.weight}'
            print('Functional user thumbs-up (weight=1.0): PASS')
    
    await test_thumbs_up_functional()

## 3. Thumbs-Up as Global Auditor (weight = 2.0)

In [4]:
if test_msg_id:
    async def test_thumbs_up_auditor():
        async with get_session() as session:
            auditor = (await session.execute(select(User).where(User.email == 'auditor@example.com'))).scalar_one()
            
            feedback = await feedback_service.record_feedback(
                session=session,
                msg_id=test_msg_id,
                rating=5,
                given_by=auditor.user_id,
                comments=None,
            )
            print(f'Auditor feedback: rating={feedback.rating}, weight={feedback.weight}')
            assert feedback.weight == 2.0, f'Expected 2.0, got {feedback.weight}'
            print('Global Auditor thumbs-up (weight=2.0): PASS')
    
    await test_thumbs_up_auditor()

## 4. Thumbs-Down Without Comment — Should Fail

In [5]:
if test_msg_id:
    async def test_thumbs_down_no_comment():
        async with get_session() as session:
            it_user = (await session.execute(select(User).where(User.email == 'it.user@example.com'))).scalar_one()
            
            try:
                await feedback_service.record_feedback(
                    session=session,
                    msg_id=test_msg_id,
                    rating=1,
                    given_by=it_user.user_id,
                    comments=None,  # No comment — should raise
                )
                print('ERROR: Should have raised FeedbackError')
            except FeedbackError as e:
                print(f'Correctly rejected: {e}')
                print('Thumbs-down requires comment: PASS')
    
    await test_thumbs_down_no_comment()

## 5. Thumbs-Down With Comment — Flags Conversation

In [6]:
if test_msg_id and test_conv_id:
    async def test_thumbs_down_flags():
        async with get_session() as session:
            it_user = (await session.execute(select(User).where(User.email == 'it.user@example.com'))).scalar_one()
            
            # Submit thumbs-down with comment
            feedback = await feedback_service.record_feedback(
                session=session,
                msg_id=test_msg_id,
                rating=1,
                given_by=it_user.user_id,
                comments='The response was missing key details about MFA exceptions.',
            )
            print(f'Thumbs-down recorded: weight={feedback.weight}')
            
            # Flag the conversation
            conv = await flag_service.flag_conversation(
                session=session,
                conv_id=test_conv_id,
                reason=f'negative_feedback: {feedback.comments}',
            )
            
            assert conv.is_flagged == True
            print(f'Conversation {test_conv_id} flagged: {conv.is_flagged}')
            print('Snapshot written to data/flagged_conversations.jsonl')
            print('Thumbs-down flags conversation: PASS')
    
    await test_thumbs_down_flags()

## 6. Weighted Feedback Aggregation

In [7]:
if test_msg_id:
    async def test_weighted_aggregation():
        async with get_session() as session:
            summary = await feedback_service.get_weighted_feedback_summary(session, test_msg_id)
            print('Weighted feedback summary:')
            print(f'  Total responses: {summary["total_responses"]}')
            print(f'  Avg rating (unweighted): {summary["avg_rating"]:.2f}')
            print(f'  Weighted avg: {summary["weighted_avg"]:.2f}')
            print('Weighted aggregation: PASS')
    
    await test_weighted_aggregation()

## Summary

Feedback & flagging validation complete:
- Functional user thumbs-up → weight=1.0 ✓
- Global Auditor thumbs-up → weight=2.0 ✓
- Thumbs-down without comment → rejected ✓
- Thumbs-down with comment → flags conversation + writes snapshot ✓
- Weighted aggregation query ✓

**Next:** Notebook 07 — Validation Harness